In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "cat_ejec_dbricks")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "saejercdatabricks")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/Reinstalaciones.csv"

In [0]:
df_reinstalaciones = (
    spark.read.option("header", True)
    .option("sep", ";")
    .option("inferSchema", True)
    .option("multiLine", True) 
    .option("quote", '"')
    .option("escape", '"') 
    .option("encoding", "ISO-8859-1") 
    .csv(ruta)
)

In [0]:
df_reinstalaciones.head(5)

In [0]:
from pyspark.sql import functions as F

columnas_fecha = [
    "FECHA GRABADA",
    "FECHA PLANIFICACION",
    "FECHA REALIZADA",
    "ULT_FECPAGO",
]

for col_name in columnas_fecha:
    col_limpia = F.regexp_replace(
        F.regexp_replace(F.col(col_name), r"[\?\.\u00A0\u202F]", ""),
        r"(?i)(am|pm)$",
        " $1",
    )
    df_reinstalaciones = df_reinstalaciones.withColumn(
        col_name,
        F.coalesce(
            F.try_to_timestamp(col_limpia, F.lit("dd/MM/yyyy h:mm:ss a")),
            F.try_to_timestamp(F.col(col_name), F.lit("yyyy-MM-dd HH:mm:ss")),
        ),
    )

In [0]:
from pyspark.sql import functions as F

columnas_texto = ["DESCRIPCION", "OBSERVACION"]

for col_name in columnas_texto:
    df_reinstalaciones = df_reinstalaciones.withColumn(
        col_name,
        F.trim(
            F.regexp_replace(
                F.col(col_name),
                r"[\r\n\t]+", " ",
            )
        ),
    )

In [0]:
df_reinstalaciones.head(5)

In [0]:
df_reinst_selecc = df_reinstalaciones.select(col("CODIGO").alias('OT_REINSTALACION'), 
                                                col("DESCRIPCION"), 
                                                col("OBSERVACION"), 
                                                col("FECHA GRABADA").alias("FECHA_GRABADA"), 
                                                col("FECHA PLANIFICACION").alias("FECHA_PLANIFICACION"),
                                                col("FECHA REALIZADA").alias("FECHA_REALIZADA"),                                                
                                                col("SUSCRIPTOR"), 
                                                col("ZONA"), 
                                                col("CICLO"), 
                                                col("ESTRATO"), 
                                                col("COBRO"),
                                                col("RESULTADO"), 
                                                col("USU_REALIZA").alias('USUARIO_REALIZA'), 
                                                col("ULT_FECPAGO").alias("FECHA_ULTIMO_PAGO"))

In [0]:
df_reinst_selecc = df_reinst_selecc.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
df_reinst_selecc.head(5)

In [0]:
df_reinst_selecc.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.reinstalaciones")

In [0]:
%sql
select  * from cat_ejec_dbricks.bronze.reinstalaciones where codigo = 543883